# Building a Bookstore DB 🤗

# Create Database and Tables

In [17]:
import sqlite3

In [2]:
# Create a database named Bookstore
conn = sqlite3.connect('Bookstore.db')
cursor = conn.cursor()

In [3]:
# Create Authors table
# Stores author information (AuthorID, Name)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Authors (
        AuthorID INTEGER PRIMARY KEY,
        Name TEXT
    )
''')
#------------------------------------------------------
# Create Books table
# Stores book information (BookID, Title, AuthorID, Price)
cursor.execute('''
    CREATE TABLE Books (
        BookID INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        AuthorID INTEGER,
        Price REAL,
        FOREIGN KEY (AuthorID) REFERENCES Authors(AuthorID)
    )
''')
#----------------------------------------------------
# Create Sales table
# Stores sales data (SaleID, BookID, Quantity, SaleDate)
cursor.execute('''
    CREATE TABLE Sales (
        SaleID INTEGER PRIMARY KEY AUTOINCREMENT,
        BookID INTEGER,
        Quantity INTEGER,
        SaleDate TEXT,
        FOREIGN KEY (BookID) REFERENCES Books(BookID)
    )
''')


In [4]:
# Commit the changes
conn.commit()
print("Table created successfully")

Table created successfully


In [6]:
# Show tables created
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('Authors',), ('Books',), ('sqlite_sequence',), ('Sales',)]


# Insert Sample Data

In [7]:
# Insert authors
authors_data = [
    ('Mohamed',),
    ('Ali',),
    ('Ahmed',),
    ('Moaz',),
    ('Fady',)
]
cursor.executemany('INSERT INTO Authors (Name) VALUES (?)', authors_data)


In [8]:
# Insert books
books_data = [
    ('Pride and Prejudice', 1, 12.99),
    ('Sense and Sensibility', 1, 10.99),
    ('A Tale of Two Cities', 2, 15.99),
    ('Oliver Twist', 2, 14.99),
    ('The Hobbit', 3, 18.99),
    ('The Lord of the Rings', 3, 25.99),
    ('Murder on the Orient Express', 4, 11.99),
    ('And Then There Were None', 4, 9.99),
    ('The Great Gatsby', 5, 13.99),
    ('Tender is the Night', 5, 12.99)
]
cursor.executemany('INSERT INTO Books (Title, AuthorID, Price) VALUES (?, ?, ?)', books_data)

In [9]:
# Insert sales data
sales_data = [
    (1, 2, '2023-01-15'),
    (1, 3, '2023-02-20'),
    (2, 1, '2023-03-10'),
    (3, 5, '2023-04-05'),
    (4, 2, '2023-05-12'),
    (5, 1, '2023-06-18'),
    (6, 4, '2023-07-22'),
    (7, 2, '2023-08-01'),
    (8, 3, '2023-09-15'),
    (9, 1, '2023-10-28'),
    (10, 2, '2023-11-11'),
    (1, 1, '2023-12-05'),
    (2, 4, '2024-01-05'),
    (8, 4, '2024-01-15'),
]
cursor.executemany('INSERT INTO Sales (BookID, Quantity, SaleDate) VALUES (?, ?, ?)', sales_data)

In [10]:
# Commit the changes
conn.commit()

print("Data inserted successfully")

Data inserted successfully


In [12]:
# Retrieve all records from the 'author' table
cursor.execute("SELECT * FROM Authors ")
rows = cursor.fetchall()

print("All records in the 'author' table:")
for row in rows:
    print(row)

All records in the 'author' table:
(1, 'Mohamed')
(2, 'Ali')
(3, 'Ahmed')
(4, 'Moaz')
(5, 'Fady')


In [15]:
# Retrieve all records from the 'Books' table
cursor.execute("SELECT * FROM Books ")
rows = cursor.fetchall()

print("All records in the 'Books' table:")
for row in rows:
    print(row)
    print("-"*60)

All records in the 'Books' table:
(1, 'Pride and Prejudice', 1, 12.99)
------------------------------------------------------------
(2, 'Sense and Sensibility', 1, 10.99)
------------------------------------------------------------
(3, 'A Tale of Two Cities', 2, 15.99)
------------------------------------------------------------
(4, 'Oliver Twist', 2, 14.99)
------------------------------------------------------------
(5, 'The Hobbit', 3, 18.99)
------------------------------------------------------------
(6, 'The Lord of the Rings', 3, 25.99)
------------------------------------------------------------
(7, 'Murder on the Orient Express', 4, 11.99)
------------------------------------------------------------
(8, 'And Then There Were None', 4, 9.99)
------------------------------------------------------------
(9, 'The Great Gatsby', 5, 13.99)
------------------------------------------------------------
(10, 'Tender is the Night', 5, 12.99)
-----------------------------------------------

In [19]:
# Retrieve all records from the 'Sales' table
cursor.execute("SELECT * FROM Sales ")
rows = cursor.fetchall()

print("All records in the 'Sales' table:")
for row in rows:
    print(row)
    print("-"*60)

All records in the 'Sales' table:
(1, 1, 2, '2023-01-15')
------------------------------------------------------------
(2, 1, 3, '2023-02-20')
------------------------------------------------------------
(3, 2, 1, '2023-03-10')
------------------------------------------------------------
(4, 3, 5, '2023-04-05')
------------------------------------------------------------
(5, 4, 2, '2023-05-12')
------------------------------------------------------------
(6, 5, 1, '2023-06-18')
------------------------------------------------------------
(7, 6, 4, '2023-07-22')
------------------------------------------------------------
(8, 7, 2, '2023-08-01')
------------------------------------------------------------
(9, 8, 3, '2023-09-15')
------------------------------------------------------------
(10, 9, 1, '2023-10-28')
------------------------------------------------------------
(11, 10, 2, '2023-11-11')
------------------------------------------------------------
(12, 1, 1, '2023-12-05')
---

# Query and Use SQL Joins

In [22]:
# Retrieve all books and their authors
import pandas as pd
first_query = """
    SELECT
        Books.Title,
        Authors.Name AS Author
    FROM Books
    INNER JOIN Authors ON Books.AuthorID = Authors.AuthorID
"""
result1_df = pd.read_sql_query(first_query, conn)
#-----------------------------------------------------
result1_df

,Title,Author
0,Pride and Prejudice,Mohamed
1,Sense and Sensibility,Mohamed
2,A Tale of Two Cities,Ali
3,Oliver Twist,Ali
4,The Hobbit,Ahmed
5,The Lord of the Rings,Ahmed
6,Murder on the Orient Express,Moaz
7,And Then There Were None,Moaz
8,The Great Gatsby,Fady
9,Tender is the Night,Fady


In [24]:
# Find total sales per book (total quantity sold)
second_query = """
    SELECT
        Books.Title,
        SUM(Sales.Quantity) AS TotalQuantitySold,
        SUM(Sales.Quantity * Books.Price) AS TotalRevenue
    FROM Books
    INNER JOIN Sales ON Books.BookID = Sales.BookID
    GROUP BY Books.BookID
"""
#-----------------------------------------------------
result2_df = pd.read_sql_query(second_query, conn)
result2_df

,Title,TotalQuantitySold,TotalRevenue
0,Pride and Prejudice,6,77.94
1,Sense and Sensibility,5,54.95
2,A Tale of Two Cities,5,79.95
3,Oliver Twist,2,29.98
4,The Hobbit,1,18.99
5,The Lord of the Rings,4,103.96
6,Murder on the Orient Express,2,23.98
7,And Then There Were None,7,69.93
8,The Great Gatsby,1,13.99
9,Tender is the Night,2,25.98


In [25]:
# Combine data from Books and Sales using JOIN
third_query = """
    SELECT
        Books.Title,
        Sales.Quantity,
        Sales.SaleDate,
        Books.Price
    FROM Books
    INNER JOIN Sales ON Books.BookID = Sales.BookID
"""
result3_df = pd.read_sql_query(third_query, conn)
#-----------------------------------------------------
result3_df

,Title,Quantity,SaleDate,Price
0,Pride and Prejudice,2,2023-01-15,12.99
1,Pride and Prejudice,3,2023-02-20,12.99
2,Sense and Sensibility,1,2023-03-10,10.99
3,A Tale of Two Cities,5,2023-04-05,15.99
4,Oliver Twist,2,2023-05-12,14.99
5,The Hobbit,1,2023-06-18,18.99
6,The Lord of the Rings,4,2023-07-22,25.99
7,Murder on the Orient Express,2,2023-08-01,11.99
8,And Then There Were None,3,2023-09-15,9.99
9,The Great Gatsby,1,2023-10-28,13.99


In [26]:
# Close the connection
conn.close()
print("Database connection closed")

Database connection closed
